<a href="https://colab.research.google.com/github/shown5/Hands-on-Generative-AI/blob/main/chap6_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

６章言語モデルのファインチューニング

- ファインチューニング済みエンコーダーモデルを用いたテキストのトピック分類
- 現代の LLM 時代におけるエンコーダーベースモデルの役割の理解
- デコーダーモデルを使った特定のスタイルのテキスト生成
- 命令型ファインチューニングによる単一モデルでの複数タスクの解決
- 小さいGPUでもモデルを訓練できるパラメーター効率の高いファインチューニング手法
- よりすくに計算資源でモデルの推論を実行できる手法

6.1.1 データセットの特定

In [1]:
%pip install genaibook

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.2 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10

In [2]:
from datasets import load_dataset

#az news データセットはテキスト分類モデルのベンチマークやデータマイニング、情報検索、データストリーミングなどの研究で広く用いられている。
# 訓練用のサンプルは１２万件あり、ファインにチューニングには十分
raw_datasets = load_dataset("fancyzhx/ag_news")
raw_datasets

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [3]:
# データの具体的な例を見ていこう

raw_train_datasets = raw_datasets["train"]
raw_train_datasets[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

//{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

サンプルにテキストとラベルが含まれているが、２はどのクラスを指しているのか？
これを知るにはデータセットの features とその label フィールドを見ればいい。

In [4]:
print(raw_train_datasets.features)

# results
# {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}
# 0なら世界のニュース、1ならスポーツ、2ならビジネス、３なら科学技術のニュース

{'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


# 6.1.2 使用するモデルタイプの定義

## Transformer おさらい

- エンコーダーモデル：入力の意味表現を捉える
- デコーダーモデル：文章などの新しいシーケンスを出力することを目的にしたモデル。テキスト生成に最適。
- エンコーダーデコーダー型モデル：入力シーケンスを異なる出力シーケンスに変換するタスクに適している

今回は、**分類ヘッド付きエンコーダーモデル**をアプローチとして採用する。
エンコーダーモデルにシンプルな分類ネットワーク（ヘッド）を埋め込みに追加してファインチューニングする方法。

ベースモデルの要件は以下の４つ

- エンコーダベースであること
- GPU を使えば数分くらいでファインチューニングできるモデル
- 事前訓練で確かな成果を残しているもの
- 短いテキストシーケンスを処理できること

DistilBERT が良さげらしい。

6.1.4 データセットの前処理

トークナイザーは AutoTokenizer を使おう。
transformers ライブラリは入力の長さがすべて同じ出なければならないので、padding=true で使おう。


In [5]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(batch):
  return tokenizer(
      batch["text"], truncation=True, padding=True, return_tensors="pt"
  )

tokenize_function(raw_train_datasets[:2])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': tensor([[  101,  2813,  2358,  1012,  6468, 15020,  2067,  2046,  1996,  2304,
          1006, 26665,  1007, 26665,  1011,  2460,  1011, 19041,  1010,  2813,
          2395,  1005,  1055,  1040, 11101,  2989,  1032,  2316,  1997, 11087,
          1011, 22330,  8713,  2015,  1010,  2024,  3773,  2665,  2153,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [  101, 18431,  2571,  3504,  2646,  3293, 13395,  1006, 26665,  1007,
         26665,  1011,  2797,  5211,  3813, 18431,  2571,  2177,  1010,  1032,
          2029,  2038,  1037,  5891,  2005,  2437,  2092,  1011, 22313,  1998,
          5681,  1032,  6801,  3248,  1999,  1996,  3639,  3068,  1010,  2038,
          5168,  2872,  1032,  2049, 29475,  2006,  2178,  2112,  1997,  1996,
          3006,  1012,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 

In [6]:
# データセット内の各要素に対して関数を並列で適用するもの

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 7600
    })
})

6.1.5 評価指標の定義

評価指標には evaluate というライブラリが使える。
文章分類の場合は以下の指標が有力な候補となる。

- 正解率
- 適合率
- 再現率
- F1 スコア

evaluate が提供する指標には compute() メソッドがある。

In [7]:
import evaluate

accuracy = evaluate.load("accuracy")
print(accuracy.description)
print(accuracy.compute(references=[0, 1, 0, 1], predictions=[1, 0, 0, 1]))


Accuracy is the proportion of correct predictions among the total number of cases processed. It can be computed with:
Accuracy = (TP + TN) / (TP + TN + FP + FN)
 Where:
TP: True positive
TN: True negative
FP: False positive
FN: False negative

{'accuracy': 0.5}


In [8]:
f1_score = evaluate.load("f1")

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)

  # 精度と F1 スコアを求める
  acc_result = accuracy.compute(references=labels, predictions=preds)
  acc = acc_result["accuracy"]

  f1_result = f1_score.compute(
      references=labels, predictions=preds, average="weighted"
  )
  f1 = f1_result["f1"]

  return {"accuracy": acc, "f1": f1}

6.1.6 モデルの訓練

DistilBERT はエンコーダーモデルなので、そのまま使うと埋め込みが得られるだけなので、分類タスクには使えない。
この埋め込みを分類ヘッドに渡す必要がある。

AutoModelForSequenceClassification でモデルを読み込み、分類ヘッドでモデルを訓練する。以下の２つの処理が行われる。
- 言語モデルのヘッドを取り外して読み込む。ここはモデルのエンコーダー部分であり、各トークンに対して埋め込みを出力する
- モデルの上にランダムに初期化された分類ヘッドを追加する。このヘッドは単なる線形層であり、プーリング埋め込みを受け取り、クラスごとの確率を出力する。

In [9]:
import torch
from transformers import AutoModelForSequenceClassification

from genaibook.core import get_device

device = get_device()
num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=num_labels
).to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


モデルの初期化ができたのでいよいよ訓練開始。

In [10]:
from transformers import TrainingArguments
from google.colab import userdata # userdataをインポート

batch_size = 32
training_args = TrainingArguments(
    "classifier-chapter4",
    push_to_hub=True, #モデルが保存されるたびに HuggingFace にプッシュするかどうか
    num_train_epochs=2, #何回転させるか
    eval_strategy="epoch", # 評価するタイミングの指定（epoch 終了時を指定）
    per_device_train_batch_size=batch_size, # 訓練時のコアあたりのバッチサイズ
    per_device_eval_batch_size=batch_size,
    hub_token=userdata.get('HF_TOKEN') # シークレットからトークンを取得して使用
)

In [11]:
#AG News データセットからデータを取得して訓練開始。

from transformers import Trainer
# huggingface_hub import notebook_login は削除。

# notebook_login() は不要になるため削除。

# データセットをシャッフルし訓練用に１万件のサンプルを抽出する
shuffled_dataset = tokenized_datasets["train"].shuffle(seed=42)
small_split = shuffled_dataset.select(range(10000))

# Trainer を初期化する
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=small_split,
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
)

/tmp/ipykernel_5881/3763507692.py:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:
import os
from google.colab import userdata

# Set the WANDB API key from Colab secrets
os.environ["WANDB_API_KEY"] = userdata.get('WANDB')

In [13]:
# trainer を初期化し訓練開始

trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: shown5 (shown5-whi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


CommError: Run initialization has timed out after 90.0 sec. Please try increasing the timeout with the `init_timeout` setting: `wandb.init(settings=wandb.Settings(init_timeout=120))`.

In [ ]:
trainer.push_to_hub()

In [ ]:
# パイプラインを使って高レベルに操作する
from transformers import pipeline

pipe = pipeline(
    "text-classification",
    model="shown5/classifier-chapter4",
    device=device
)
pipe(
    """The soccer match between Sapin and Portugal ended in a terrible result for Portugal."""
)

In [ ]:
# 予測結果の評価

# すべてのサンプルについて予測を得る
model_preds = pipe(list(tokenized_datasets["test"]["text"])) # ここを修正

# データセットのラベルを得る
references = tokenized_datasets["test"]["label"]


# ラベルのリストを得る
label_names = raw_train_datasets.features["label"].names

# 最初の３サンプルの結果を表示する
samples = 3
texts = tokenized_datasets["test"]["text"][:samples]
for pred, ref, text in zip(model_preds[:samples], references[:samples], texts):
  print(f"Predicted: {pred['label']}; Actual {label_names[ref]};")
  print(text)

In [ ]:
# 混同行列（真陽性偽陽性など）でモデルの性能を確認する

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# 予測されたラベルをIDに変換する
label_to_id = {name: i for i, name in enumerate(label_names)}
preds_labels = [int(pred["label"].split('_')[1]) for pred in model_preds] # ここを修正

#混同行列を求める
confusion_matrix = evaluate.load("confusion_matrix")
cm = confusion_matrix.compute(
    references=references,
    predictions=preds_labels,
    normalize="true"
)["confusion_matrix"]

# 混同行列をプロットする
fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
plt.title("Normalised confusion matrix")
plt.show()

6.1.7 今でも役に立つのか

以前として小型のカスタム分類機が役に立つことはある。とりわけ速度と効率が重要なアプリケーションでは。
例えば超大規模言語モデルの訓練データの準備で有用。大量の訓練データを大規模モデルに投入するのはコストが高い。
また、検索システム向けに埋め込みを得ることにも使える。
とはいえ、やはり高性能モデルの利用に移行しつつあるのが現状。

6.2 テキスト生成

In [ ]:
filtered_datasets = raw_datasets.filter(lambda example: example["label"] == 2)
filtered_datasets = filtered_datasets.remove_columns("label")

6.2.1 適切な生成モデルの選択

どのベースモデルを使うのかを判断するための要素は次のとおり。

- モデルサイズ：デカすぎても小さすぎてもダメ。
- 訓練データ：推論時のデータと類似している訓練データを選定すべし
- コンテキスト長：長文を生成する際にはコンテキストも長いものにする必要がある
- ライセンス：商用OKかどうかは確認すべし


6.2.2 生成モデルの訓練

In [ ]:
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

model_id = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = (
    tokenizer.eos_token
)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

In [ ]:
# データセットをトークン化する

def tokenize_function(batch):
  return tokenizer(batch["text"], truncation=True)

tokenized_datasets = filtered_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"], # input_ids と attention_mask のみ必要
)

In [ ]:
tokenized_datasets

In [ ]:
# 因果言語モデリングようのデータコレーターを作成する

from transformers import DataCollatorForLanguageModeling

# mlm はマスク言語モデルの略
# 今回はマスク言語モデルを訓練するわけではなく、因果言語モデル（causal language model）を訓練するためにFalseに設定する
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
samples = [tokenized_datasets["train"][i] for i in range(3)]
for sample in samples:
  print(f"input_ids shape: {len(sample['input_ids'])}")

In [ ]:
out = data_collator(samples)
for key in out:
  print(f"{key} shape : {out[key].shape}")

In [ ]:
training_args = TrainingArguments(
    "business-news-generator",
    push_to_hub=True,
    per_device_train_batch_size=8,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    num_train_epochs=2,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=200
)

In [ ]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"].select(range(5000)),
    eval_dataset=tokenized_datasets["test"],
)

In [ ]:
trainer.train()

In [ ]:
trainer.push_to_hub()

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="shown5/business-news-generator",
    device=device
)
print(
    pipe("Q1", do_sample=True, temperature=0.1, max_new_tokens=30)[0]["generated_text"]
)
print(
    pipe("Wall", do_sample=True, temperature=0.1, max_new_tokens=30)[0]["generated_text"]
)
print(
    pipe("Google", do_sample=True, temperature=0.1, max_new_tokens=30)[0]["generated_text"]
)

6.3 インストラクション

モデルを用途ごとにファインチューニングする方法について学んだが、これだとタスクごとにファインチューニングが必要になるので効率が良くないことがある。

アダプター：ベースモデルを凍結したまま、小規模なアダプターのみを訓練する方法。次節で詳細を実施。

6.4 アダプターの概要

PEFT（Parameter-Efficient Fine Tuning）パラメーター効率的ファインチューニングという手法で、アダプターを使う。モデルを訓練するのではなく、少数の追加パラメーターをモデルに加える方法。

PEFT にはさまざまな手法があり代表的なのは「Prefix Tuning」「Prompt Tuning」「Low-Rank Adaptation」がある。

ここでは LoRA を取り上げる。LoRA は重みの更新を更新行列と呼ばれる２つの小さな行列によって低ランク分解で表現する手法。

LoRA のコードを見ていく。

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM"
)

model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M")
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

6.5 量子化の簡単な紹介

PEFT を使えば低コストで訓練ができるものの、推論時のモデルのサイズは変わらない。
推論時のモデルのパラメーターは FP16 でもOK（FP32 より精度は低いが推論時には 16 で十分なので）。
70 億のパラメーターの場合、必要なメモリサイズは 32 だと 280 億バイト（26GB）だが 16 だと 13GB まで小さくできる。

pytorch はデフォルトで 32 を使う。

もっと精度を落とす方法として 8ビット量子化 がある。
単純な absmax 量子化を取り上げる。
これは最大値をまず取得して、この最大値を 127（取りうる最大の値）で割る。これで量子化係数というものが得られる。この係数をベクトルにかけることで最大値が 127 になることが保証される。

実際にコードで見てみる。

In [ ]:
import numpy as np

def scaling_factor(vector):
  # ベクトルの最大値を取る
  m = np.max(np.abs(vector))

  # 係数を求める
  return 127 / m

array = [1.2, -0.5, -4.3, 1.2, -3.1, 0.8, 2.4, 5.4, 0.3]
alpha = scaling_factor(array)
quantized_array = np.round(alpha * np.array(array)).astype(np.int8)
dequantized_array = quantized_array / alpha

print(f"Scaling factor: {alpha}")
print(f"Quantized array: {quantized_array}")
print(f"Dequantized array: {dequantized_array}")
print(f"Difference: {array - dequantized_array}")]

結果を見ると、まずまずの差異ができるため性能劣化が起きがちだった。
これに対処できる方法として LLM.in8() がある。

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    quantization_config=quantization_config
)

6.6 全てをまとめる

- PEFT はアダプターを追加しベースモデルの重みを凍結することではるかに少ない計算量でモデルのファインチューニングを可能にする。これにより更新される重みがごく一部で済むため訓練が高速になる
- 量子化は保存時に使われているビット数より少ないビット数でモデルをロードできるようにする。これによりモデルのロードや推論に必要なGPUの要件を下げることができる。

両方を組み合わせたものが QLoRA。

In [ ]:
%pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.3",
    quantization_config=quantization_config,
    device_map="auto"
)

In [ ]:
from trl import SFTConfig, SFTTrainer

dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")

peft_config = LaraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

sft_cofig = SFTConfig(
    "fine_tune_e2e",
    push_device_train_batch_size=8,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    num_train_epochs=2,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=200,
    gradient_checkpointing=True,
    max_seq_length=512,
    dataset_text_field="text",
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset.select(range(300)),
    peft_config=peft_config,
)

trainer.train()

In [ ]:
trainer.push_to_hub()

In [ ]:
# ベースモデルを読み込む
tokenizer = AutoTokenizer.from_pretrained("mistral/Mistral-7B-v0.3")
model = AutoModelForCausalLM.from_pretrained(
    "mistral/Mistral-7B-v0.3",
    torch_dtype=torch.float16,
    device_map="auto"
)

# load-adaptor でアダプターを読み込む
model.load_adapter("shown5/fine_tune_e2e")
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
pipe("### Human: Hello!### Assistant:", max_new_tokens=100)

In [ ]:
pipe = pipeline(
    "text-generation",
    "HuggingFaceTB/SmolLM-135M-Instruct",
    device=device
)
messages = [
    {
        "role": "system",
        "content": """You are a friendly chatbot who always responds in the style of a pirate""",
    },
    {
        "role": "user",
        "content": "How many helicopters can a human eat in one sitting?",
    }
]
print(pipe(messages, max_new_tokens=128)[0]["generated_text"][-1])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M-Instruct")

chat = [
    {
        "role": "user", "content": "Hello, how are you?"
    },
    {
        "role": "assistant", "content": "I'm good, thanks!"
    },
    {
        "role": "user", "content": "I'd like to show off how chat tempting works!"
    }
]

tokenizer.apply_chat_template(chat, tokenizer=False)

print(tokenizer.apply_chat_template(chat, tokenize=False))

6.7 評価の深掘り